In [9]:
pip install rioxarray matplotlib_scalebar

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# **EXTRACTION**

In [10]:
"""
Skrip Ekstraksi Lahan Sawah dari Dynamic World (Google Earth Engine)
+ Visualisasi Peta Kartografis
=====================================================================
Alur kerja dalam satu skrip:
  1. Baca AOI dari file SHP milik user
  2. Ambil citra Dynamic World dari (TANGGAL_PILIHAN - RENTANG_BULAN) s/d TANGGAL_PILIHAN
  3. Ekstraksi kelas Sawah (Flooded Vegetation + Crops)
  4. Ekspor hasil: COG (GeoTIFF) + GeoJSON  <-- satu-satunya output data
  5. Susun peta kartografis (judul, arah utara, skala, legenda, grid, garis tepi) -> PNG

Dependensi:
  earthengine-api, geopandas, rioxarray, rasterio, shapely, pandas, numpy,
  requests, pyproj, matplotlib, matplotlib-scalebar

Instalasi tambahan yang mungkin diperlukan:
  pip install matplotlib-scalebar --break-system-packages
"""

import os
import zipfile
import warnings
import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import xarray as xr
from rasterio.features import shapes
from shapely.geometry import shape
from pyproj import CRS
import requests
import ee

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.ndimage import convolve

# --- OPSIONAL: majority filter pasca-ekstraksi ---
TERAPKAN_MAJORITY_FILTER = True
UKURAN_JENDELA_MAJORITY = 5

warnings.filterwarnings('ignore')

# ==============================================================================
# 0. KONFIGURASI — EDIT BAGIAN INI SESUAI KEBUTUHAN
# ==============================================================================
PROJECT_ID = "poetic-emblem-487109-t6"

# --- Struktur folder proyek (paddy/data/raw, paddy/data/processed, paddy/results/figures) ---
RAW_DIR = "../../data/raw"
PROCESSED_DIR = "../../data/processed"
FIGURES_DIR = "../../results/figures/module_1"
MODULE_DIR = os.path.join(PROCESSED_DIR, "module_1")
os.makedirs(MODULE_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- 1) INPUT AOI MANUAL (Shapefile) ---
# Path ke file .shp (pastikan file .shx, .dbf, .prj berada di folder yang sama)
SHP_PATH = os.path.join(RAW_DIR, "AOI_1.shp")

# --- 2) INPUT TANGGAL MANUAL (cukup SATU tanggal) ---
# Skrip otomatis mencari komposit mode Dynamic World dari (TANGGAL_PILIHAN - RENTANG_BULAN)
# sampai TANGGAL_PILIHAN.
TANGGAL_PILIHAN = "2026-08-31"   # format: YYYY-MM-DD
RENTANG_BULAN = 12                # jumlah bulan mundur dari TANGGAL_PILIHAN

# --- OPSIONAL: paksa sistem proyeksi UTM tertentu ---
# Kosongkan (None) agar dideteksi otomatis berdasarkan lokasi AOI
EPSG_UTM_MANUAL = None   # contoh: 32749 untuk UTM zona 49S

RESOLUSI_M = 10

# Kode kelas Dynamic World yang dianggap "Sawah"
# 3 = Flooded Vegetation, 4 = Crops
KODE_SAWAH = [3, 4]
NODATA_KELAS = 255

# --- OUTPUT DATA (hanya dua file ini yang dihasilkan) ---
OUTPUT_COG = os.path.join(MODULE_DIR, "Klasifikasi_Sawah.tif")
OUTPUT_GEOJSON = os.path.join(MODULE_DIR, "Klasifikasi_Sawah.geojson")

# --- KONFIGURASI PETA KARTOGRAFIS ---
BUAT_PETA = True                                   # set False jika tidak ingin membuat peta
OUTPUT_PETA_PNG = os.path.join(FIGURES_DIR, "Peta_Sawah_Kartografis.png")
JUDUL_PETA = "PETA SEBARAN LAHAN SAWAH"
SUB_JUDUL = "Hasil Klasifikasi Citra Dynamic World V1"
SUMBER_DATA = "Google Dynamic World V1 (Google Earth Engine)"
PEMBUAT_PETA = "Diolah oleh: [PT. Herdento Global Solusi]"
DPI_PETA = 300
FIGSIZE_INCI = (11, 9)          # ukuran kertas cetak (lebar, tinggi) dalam inci — memengaruhi hitungan skala
WARNA_SAWAH = "#2CA25F"
WARNA_NON_SAWAH = "#EDEDED"
TAMPILKAN_BATAS_AOI = True       # overlay batas AOI (dari SHP_PATH) sebagai garis putus-putus di peta


# ==============================================================================
# 1. AUTENTIKASI GEE
# ==============================================================================
def init_gee():
    print("=== TAHAP 1: Inisialisasi Google Earth Engine ===")
    try:
        ee.Initialize(project=PROJECT_ID)
        print(f"Berhasil terhubung ke GEE (Project ID: {PROJECT_ID})")
    except Exception:
        print("Membutuhkan autentikasi awal...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)


# ==============================================================================
# 2. BACA AOI DARI SHP USER
# ==============================================================================
def baca_aoi_dari_shp(shp_path, epsg_utm_manual=None):
    print(f"\n=== TAHAP 2: Membaca AOI dari file SHP: {shp_path} ===")
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"File SHP tidak ditemukan: {shp_path}")

    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        raise ValueError("Shapefile tidak memiliki sistem proyeksi (CRS). Pastikan file .prj tersedia.")

    # Gabungkan seluruh fitur menjadi satu geometri AOI (jika SHP berisi banyak polygon)
    geom_union = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
    gdf_aoi = gpd.GeoDataFrame(geometry=[geom_union], crs=gdf.crs)

    # Tentukan sistem proyeksi UTM (otomatis atau manual)
    if epsg_utm_manual:
        epsg_utm = epsg_utm_manual
    else:
        epsg_utm = gdf_aoi.estimate_utm_crs().to_epsg()
    print(f"-> Menggunakan sistem proyeksi UTM: EPSG:{epsg_utm}")

    gdf_aoi_utm = gdf_aoi.to_crs(f"EPSG:{epsg_utm}")
    bbox_wgs84 = tuple(gdf_aoi.to_crs("EPSG:4326").total_bounds)

    luas_ha = gdf_aoi_utm.geometry.area.iloc[0] / 10000
    print(f"-> Luas AOI: {luas_ha:,.2f} Ha")

    return bbox_wgs84, gdf_aoi_utm, epsg_utm


# ==============================================================================
# 3. PROSES DYNAMIC WORLD DARI GEE (SATU TANGGAL -> RENTANG N BULAN KE BELAKANG)
# ==============================================================================
def hitung_rentang_tanggal(tanggal_pilihan, rentang_bulan):
    """Hitung tanggal_akhir = tanggal_pilihan, tanggal_mulai = tanggal_pilihan - rentang_bulan."""
    tanggal_akhir_ee = ee.Date(tanggal_pilihan)
    tanggal_mulai_ee = tanggal_akhir_ee.advance(-rentang_bulan, 'month')

    tanggal_mulai_str = tanggal_mulai_ee.format('YYYY-MM-dd').getInfo()
    tanggal_akhir_str = tanggal_akhir_ee.format('YYYY-MM-dd').getInfo()
    return tanggal_mulai_ee, tanggal_akhir_ee, tanggal_mulai_str, tanggal_akhir_str


def proses_dynamic_world(bbox_wgs84, gdf_aoi_utm, epsg_utm, tanggal_pilihan, rentang_bulan):
    tanggal_mulai_ee, tanggal_akhir_ee, tanggal_mulai_str, tanggal_akhir_str = hitung_rentang_tanggal(
        tanggal_pilihan, rentang_bulan
    )
    print(f"\n=== TAHAP 3: Mengunduh Data Dynamic World ({tanggal_mulai_str} s/d {tanggal_akhir_str}) ===")
    print(f"-> Tanggal acuan: {tanggal_pilihan}, mundur {rentang_bulan} bulan")

    aoi_ee = ee.Geometry.Rectangle([bbox_wgs84[0], bbox_wgs84[1], bbox_wgs84[2], bbox_wgs84[3]])

    dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
          .filterBounds(aoi_ee)
          .filterDate(tanggal_mulai_ee, tanggal_akhir_ee))

    jumlah_citra = dw.size().getInfo()
    if jumlah_citra == 0:
        raise ValueError(
            f"Tidak ada citra Dynamic World yang tersedia antara {tanggal_mulai_str} dan {tanggal_akhir_str} "
            "pada AOI ini. Coba pilih TANGGAL_PILIHAN lain, perbesar RENTANG_BULAN, "
            "atau periksa kembali lokasi SHP."
        )
    print(f"-> Jumlah citra ditemukan pada rentang tanggal: {jumlah_citra}")

    dw_mode = dw.select('label').mode().unmask(NODATA_KELAS).clip(aoi_ee)

    url = dw_mode.getDownloadURL({
        'scale': RESOLUSI_M,
        'crs': f'EPSG:{epsg_utm}',
        'region': aoi_ee,
        'format': 'GEO_TIFF'
    })

    print("Mengunduh data citra dari server Google...")
    r = requests.get(url)

    if r.content.startswith(b'PK'):
        with open("temp_dw.zip", "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile("temp_dw.zip", "r") as z:
            z.extractall("temp_dw_dir")
        tif_path = [os.path.join("temp_dw_dir", x) for x in os.listdir("temp_dw_dir") if x.endswith(".tif")][0]
        da = rioxarray.open_rasterio(tif_path).squeeze(drop=True)
    else:
        with open("temp_dw.tif", "wb") as f:
            f.write(r.content)
        da = rioxarray.open_rasterio("temp_dw.tif").squeeze(drop=True)

    da = da.astype(np.uint8)
    da_clipped = da.rio.write_nodata(NODATA_KELAS).rio.clip(gdf_aoi_utm.geometry, gdf_aoi_utm.crs, drop=True)
    return da_clipped, tanggal_mulai_str, tanggal_akhir_str


# ==============================================================================
# 4. EKSTRAKSI KELAS SAWAH
# ==============================================================================
def ekstrak_sawah(da_kelas, kode_sawah):
    print("\n=== TAHAP 4: Ekstraksi Kelas Sawah (Flooded Vegetation + Crops) ===")

    arr = da_kelas.values
    mask_sawah = np.where(np.isin(arr, kode_sawah), 1, 0).astype(np.uint8)
    mask_sawah[arr == NODATA_KELAS] = NODATA_KELAS

    if TERAPKAN_MAJORITY_FILTER:
        print(f"-> Menerapkan majority filter {UKURAN_JENDELA_MAJORITY}x{UKURAN_JENDELA_MAJORITY}...")
        mask_sawah = majority_filter_biner(
            mask_sawah, nodata=NODATA_KELAS, ukuran_jendela=UKURAN_JENDELA_MAJORITY
        )

    luas_ha = ((mask_sawah == 1).sum() * (RESOLUSI_M ** 2)) / 10000
    print(f"-> Total Luas Sawah: {luas_ha:,.2f} Hektar")

    da_sawah = xr.DataArray(mask_sawah, dims=da_kelas.dims, coords=da_kelas.coords)
    da_sawah = da_sawah.rio.write_crs(da_kelas.rio.crs).rio.write_nodata(NODATA_KELAS)

    return da_sawah, luas_ha

def majority_filter_biner(mask_biner, nodata=NODATA_KELAS, ukuran_jendela=5):
    """
    Majority filter (mode) untuk raster biner (0/1) memakai jendela persegi
    berukuran ukuran_jendela x ukuran_jendela. Piksel NoData tidak ikut
    dihitung sebagai tetangga, dan posisinya tetap NoData pada hasil akhir.
    """
    valid = (mask_biner != nodata).astype(np.uint8)
    nilai = np.where(valid == 1, mask_biner, 0).astype(np.uint8)

    kernel = np.ones((ukuran_jendela, ukuran_jendela), dtype=np.uint8)

    jumlah_1 = convolve(nilai, kernel, mode="constant", cval=0)
    jumlah_valid = convolve(valid, kernel, mode="constant", cval=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        proporsi = np.where(jumlah_valid > 0, jumlah_1 / jumlah_valid, 0)

    hasil = np.where(proporsi >= 0.5, 1, 0).astype(np.uint8)
    hasil[mask_biner == nodata] = nodata  # kembalikan nodata ke posisi semula
    return hasil

# ==============================================================================
# 5. SIMPAN OUTPUT: COG + GEOJSON
# ==============================================================================
def simpan_cog(da_sawah, output_path):
    print("\n=== TAHAP 5: Menyimpan Raster sebagai Cloud Optimized GeoTIFF (COG) ===")
    da_sawah.rio.to_raster(
        output_path,
        driver="COG",
        compress="DEFLATE",
        overview_resampling="nearest",
        dtype="uint8"
    )
    print(f"-> COG tersimpan: {output_path}")


def simpan_geojson(da_sawah, epsg_utm, output_path):
    print("\n=== TAHAP 6: Vektorisasi Sawah & Non-Sawah -> GeoJSON ===")

    arr = da_sawah.values
    transform = da_sawah.rio.transform()

    # Vektorkan SEMUA piksel valid (0 = non-sawah, 1 = sawah), kecuali nodata
    mask_valid = arr != NODATA_KELAS

    label_kelas = {
        1: "Sawah",
        0: "Penutup/Penggunaan Lahan Non Sawah",
    }

    hasil_shapes = shapes(arr, mask=mask_valid, transform=transform)

    geometri, kelas_list, kode_list = [], [], []
    for geom, val in hasil_shapes:
        val = int(val)
        if val in label_kelas:
            geometri.append(shape(geom))
            kelas_list.append(label_kelas[val])
            kode_list.append(val)

    if not geometri:
        print("-> Peringatan: Tidak ditemukan area valid pada AOI/rentang tanggal ini. GeoJSON kosong dibuat.")

    gdf_sawah_utm = gpd.GeoDataFrame(
        {"kelas": kelas_list, "kode": kode_list},
        geometry=geometri,
        crs=f"EPSG:{epsg_utm}",
    )
    if not gdf_sawah_utm.empty:
        gdf_sawah_utm["luas_ha"] = gdf_sawah_utm.geometry.area / 10000

    gdf_sawah_wgs84 = gdf_sawah_utm.to_crs("EPSG:4326")
    gdf_sawah_wgs84.to_file(output_path, driver="GeoJSON")
    print(f"-> GeoJSON tersimpan: {output_path} ({len(gdf_sawah_wgs84)} poligon)")

    return gdf_sawah_utm  # sekarang berisi dua kelas: Sawah & Non-Sawah


# ==============================================================================
# 6. ELEMEN KARTOGRAFIS
# ==============================================================================
def tambah_arah_mata_angin(ax, x=0.94, y=0.90, ukuran=0.045):
    """Menggambar panah arah utara sederhana di pojok kanan atas peta."""
    ax.annotate(
        "U",
        xy=(x, y + ukuran), xycoords="axes fraction",
        xytext=(x, y - ukuran), textcoords="axes fraction",
        ha="center", va="center", fontsize=13, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="black", lw=2),
        zorder=10,
    )


def tambah_grid_koordinat(ax, bounds):
    """Menambahkan grid koordinat (graticule) dengan label pada keempat sisi peta."""
    lebar = bounds[2] - bounds[0]
    tinggi = bounds[3] - bounds[1]
    dim_terpanjang = max(lebar, tinggi)

    kandidat_interval = [100, 250, 500, 1000, 2000, 2500, 5000, 10000, 20000]
    interval = next((i for i in kandidat_interval if dim_terpanjang / i <= 8), kandidat_interval[-1])

    ax.xaxis.set_major_locator(MultipleLocator(interval))
    ax.yaxis.set_major_locator(MultipleLocator(interval))

    formatter = FuncFormatter(lambda v, pos: f"{v:,.0f}".replace(",", "."))
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)

    ax.tick_params(top=True, right=True, labeltop=True, labelright=True, labelsize=8)
    ax.grid(True, which="major", linestyle=":", linewidth=0.6, color="black", alpha=0.5, zorder=5)
    ax.set_xlabel("Koordinat Timur (m)", fontsize=9)
    ax.set_ylabel("Koordinat Utara (m)", fontsize=9)


def tambah_skala(ax, bounds, figsize_inci):
    """Menambahkan skala grafis (ScaleBar) dan estimasi skala angka pada ukuran cetak sebenarnya."""
    scalebar = ScaleBar(
        dx=1, units="m", location="lower left",
        length_fraction=0.25, box_alpha=0.8, color="black",
        font_properties={"size": 9}, border_pad=0.6, pad=0.4,
    )
    ax.add_artist(scalebar)

    lebar_peta_m = bounds[2] - bounds[0]
    lebar_kertas_cm = figsize_inci[0] * 2.54
    penyebut_skala = (lebar_peta_m * 100) / lebar_kertas_cm
    teks_skala = f"Skala ± 1 : {penyebut_skala:,.0f}".replace(",", ".")
    ax.text(
        0.02, 0.045, teks_skala, transform=ax.transAxes,
        fontsize=8, style="italic", ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.75),
    )


def tambah_legenda(ax, luas_sawah_ha):
    patches = [
        mpatches.Patch(color=WARNA_SAWAH, label=f"Sawah ({luas_sawah_ha:,.2f} Ha)".replace(",", ".")),
        mpatches.Patch(color=WARNA_NON_SAWAH, label="Non-Sawah"),
    ]
    ax.legend(
        handles=patches, loc="upper left", title="Legenda",
        fontsize=9, title_fontsize=10, framealpha=0.9, borderpad=0.8,
    )


def tambah_info_teks(fig, epsg, sumber, pembuat, tanggal_mulai_str, tanggal_akhir_str):
    nama_proyeksi = CRS.from_epsg(epsg).name
    tanggal_dibuat = datetime.date.today().strftime("%d-%m-%Y")

    teks_info = (
        f"Sistem Proyeksi : {nama_proyeksi} (EPSG:{epsg})\n"
        f"Sumber Data      : {sumber}\n"
        f"Rentang Citra    : {tanggal_mulai_str} s/d {tanggal_akhir_str}\n"
        f"{pembuat} | Tanggal cetak: {tanggal_dibuat}"
    )
    fig.text(0.02, 0.028, teks_info, fontsize=7.5, ha="left", va="bottom", family="monospace")


def tambah_garis_tepi(fig):
    """Membuat bingkai ganda (garis tepi) khas peta kartografis di sekeliling figure."""
    for offset, lw in [(0.012, 1.5), (0.020, 0.8)]:
        rect = mpatches.Rectangle(
            (offset, offset), 1 - 2 * offset, 1 - 2 * offset,
            transform=fig.transFigure, fill=False, edgecolor="black", linewidth=lw, zorder=20,
        )
        fig.add_artist(rect)


# ==============================================================================
# 7. SUSUN PETA KARTOGRAFIS
# ==============================================================================
def buat_peta_kartografis(da_sawah, gdf_sawah_utm, gdf_aoi_utm, epsg_utm,
                           luas_ha, tanggal_mulai_str, tanggal_akhir_str):
    print("\n=== TAHAP 7: Menyusun Peta Kartografis ===")

    arr = da_sawah.values
    bounds = da_sawah.rio.bounds()  # (left, bottom, right, top)

    fig, ax = plt.subplots(figsize=FIGSIZE_INCI)
    fig.subplots_adjust(left=0.09, right=0.93, top=0.86, bottom=0.13)

    cmap = ListedColormap([WARNA_NON_SAWAH, WARNA_SAWAH])
    norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)
    plot_arr = np.ma.masked_equal(arr, NODATA_KELAS)
    extent = (bounds[0], bounds[2], bounds[1], bounds[3])
    ax.imshow(plot_arr, cmap=cmap, norm=norm, extent=extent, origin="upper", zorder=1)

    if gdf_sawah_utm is not None and not gdf_sawah_utm.empty:
        gdf_sawah_saja = gdf_sawah_utm[gdf_sawah_utm["kode"] == 1]
    if not gdf_sawah_saja.empty:
        gdf_sawah_saja.boundary.plot(ax=ax, color="#1B5E3A", linewidth=0.4, zorder=3)

    if TAMPILKAN_BATAS_AOI and gdf_aoi_utm is not None and not gdf_aoi_utm.empty:
        gdf_aoi_utm.boundary.plot(ax=ax, color="black", linewidth=1.2, linestyle="--", zorder=4)

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_aspect("equal")

    tambah_grid_koordinat(ax, bounds)
    tambah_arah_mata_angin(ax)
    tambah_skala(ax, bounds, FIGSIZE_INCI)
    tambah_legenda(ax, luas_ha)

    fig.suptitle(JUDUL_PETA, fontsize=16, fontweight="bold", y=0.965)
    fig.text(0.5, 0.925, SUB_JUDUL, fontsize=11, ha="center", style="italic")

    tambah_info_teks(fig, epsg_utm, SUMBER_DATA, PEMBUAT_PETA, tanggal_mulai_str, tanggal_akhir_str)
    tambah_garis_tepi(fig)

    plt.savefig(OUTPUT_PETA_PNG, dpi=DPI_PETA, facecolor="white")
    plt.close(fig)
    print(f"-> Peta kartografis tersimpan: {OUTPUT_PETA_PNG}")


# ==============================================================================
# 8. MAIN PIPELINE
# ==============================================================================
def main():
    init_gee()
    bbox_wgs84, gdf_aoi_utm, epsg_utm = baca_aoi_dari_shp(SHP_PATH, EPSG_UTM_MANUAL)
    kelas_lahan, tanggal_mulai_str, tanggal_akhir_str = proses_dynamic_world(
        bbox_wgs84, gdf_aoi_utm, epsg_utm, TANGGAL_PILIHAN, RENTANG_BULAN
    )
    da_sawah, luas_ha = ekstrak_sawah(kelas_lahan, KODE_SAWAH)

    simpan_cog(da_sawah, OUTPUT_COG)
    gdf_sawah_utm = simpan_geojson(da_sawah, epsg_utm, OUTPUT_GEOJSON)

    if BUAT_PETA:
        buat_peta_kartografis(
            da_sawah, gdf_sawah_utm, gdf_aoi_utm, epsg_utm,
            luas_ha, tanggal_mulai_str, tanggal_akhir_str
        )

    print("\n=== SELESAI ===")
    print(f"Total Luas Sawah : {luas_ha:,.2f} Ha")
    print(f"Output COG       : {OUTPUT_COG}")
    print(f"Output GeoJSON   : {OUTPUT_GEOJSON}")
    if BUAT_PETA:
        print(f"Output Peta      : {OUTPUT_PETA_PNG}")


if __name__ == "__main__":
    main()

=== TAHAP 1: Inisialisasi Google Earth Engine ===
Berhasil terhubung ke GEE (Project ID: poetic-emblem-487109-t6)

=== TAHAP 2: Membaca AOI dari file SHP: ../../data/raw\AOI_1.shp ===
-> Menggunakan sistem proyeksi UTM: EPSG:32749
-> Luas AOI: 24,377.66 Ha

=== TAHAP 3: Mengunduh Data Dynamic World (2025-08-31 s/d 2026-08-31) ===
-> Tanggal acuan: 2026-08-31, mundur 12 bulan
-> Jumlah citra ditemukan pada rentang tanggal: 39
Mengunduh data citra dari server Google...

=== TAHAP 4: Ekstraksi Kelas Sawah (Flooded Vegetation + Crops) ===
-> Menerapkan majority filter 5x5...
-> Total Luas Sawah: 9,977.46 Hektar

=== TAHAP 5: Menyimpan Raster sebagai Cloud Optimized GeoTIFF (COG) ===
-> COG tersimpan: ../../data/processed\Klasifikasi_Sawah.tif

=== TAHAP 6: Vektorisasi Sawah & Non-Sawah -> GeoJSON ===
-> GeoJSON tersimpan: ../../data/processed\Klasifikasi_Sawah.geojson (1549 poligon)

=== TAHAP 7: Menyusun Peta Kartografis ===
-> Peta kartografis tersimpan: ../../results/figures/module_1\Pe